# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will load the Croissant schema, review record sets and fields by their `@id`, extract data, process and visualize it, following best practices for reproducible and FAIR data science workflows.

### Dataset Source
The dataset source is defined by a public Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This will provide an object representing all the structure of the dataset, including record set, field, and column definitions.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Let's list available Record Sets, their Fields (columns), and reference their `@id`s.

> All references to schema entities (record set, fields, columns) use the `@id` as required for FAIR and Croissant best practices.

In [ ]:
# List out record sets and their fields, referencing by their @id.
record_sets = metadata.record_sets  # List of RecordSet objects
print(f"Number of record sets: {len(record_sets)}\n")

record_set_ids = []

for idx, rs in enumerate(record_sets):
    print(f"Record Set {idx+1}: \n  name: {rs.name}\n  @id: {rs.id}\n  description: {rs.description if hasattr(rs,'description') else ''}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        desc = getattr(field, 'description', '')
        print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}] {desc}")
    print("")

# For the rest of the notebook, we'll pick the first record set to demonstrate data access.
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None


## 3. Data Extraction
Load rows from a specific record set using its `@id`, load them into a DataFrame, and show columns and some sample data.

You can adapt this to other record sets by adjusting the `record_set_id` variable.

In [ ]:
# Load data for all record sets present

dataframes = {}
for rec_id in record_set_ids:
    print(f"Loading data for RecordSet: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"  DataFrame shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print("")

# Show first 5 rows for the example record set
if example_record_set_id:
    print(f"\nSample rows for Record Set '{example_record_set_id}':")
    display(dataframes[example_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data processing, such as filtering, normalization, and grouping. All columns referenced will use the `@id` (as shown above in the Data Overview step).

**Note:** You may need to adjust `numeric_field` or `group_field` to match available `@id`s for fields in the dataset. See previous outputs for valid ids.

For illustration, let's use the following field IDs if available (otherwise update them as appropriate):
- A numeric field with protein abundance, MW, or score (e.g., `cr:abundance`, `cr:MW`, etc.)
- A possible group field like a sample or protein classification (e.g., `cr:sample`, `cr:class`, etc.).

In [ ]:
# Example EDA: Filtering, normalization, grouping

# Replace these IDs using the printout above if necessary for your dataset
numeric_field_id = None
group_field_id = None

# Try to find a numeric field and a group field
fields = None
for rs in record_sets:
    if rs.id == example_record_set_id:
        fields = rs.fields
        break

# Auto-detect numeric and group field
if fields:
    for field in fields:
        # Use first float/int field for numeric_field
        if numeric_field_id is None and field.data_type in ["schema:Float", "schema:Integer", "schema:Number"]:
            numeric_field_id = field.id
        # Use first string/categorical field as a group field
        if group_field_id is None and field.data_type in ["schema:Text", "schema:String"]:
            group_field_id = field.id

if not numeric_field_id:
    raise ValueError("Could not automatically detect a numeric field id. Please set manually from data overview.")

# Proceed with analysis
df = dataframes[example_record_set_id].copy()

# Some records may have missing or non-numeric data, so let's coerce to numeric and drop NA
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
df_filtered = df[df[numeric_field_id].notna()]  # drop NA values

threshold = df_filtered[numeric_field_id].mean()  # Use mean as example threshold
extracted = df_filtered[df_filtered[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {extracted.shape[0]} rows")
display(extracted.head())

# Normalize the numeric field (z-score)
extracted[f"{numeric_field_id}_normalized"] = (extracted[numeric_field_id] - extracted[numeric_field_id].mean()) / extracted[numeric_field_id].std()
print(f"\nFirst few normalized values for {numeric_field_id}:")
display(extracted[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id, if exists
if group_field_id and group_field_id in extracted.columns:
    grouped = extracted.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped.head())


## 5. Visualization
Now let's visualize the distribution of the selected numeric field, and if applicable, display grouped means.

We will use matplotlib and seaborn for plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(extracted[numeric_field_id], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id and group_field_id in extracted.columns:
    plt.figure(figsize=(10,4))
    # Show top categories only if there are too many
    top_groups = extracted[group_field_id].value_counts().head(10).index.tolist()
    subset = extracted[extracted[group_field_id].isin(top_groups)]
    sns.boxplot(data=subset, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id} (Top 10)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated a reproducible workflow for:
- Loading and inspecting a Croissant-based dataset using `mlcroissant`
- Listing all record sets and their fields by `@id`
- Extracting and analyzing record set contents in DataFrames
- Filtering, normalizing, and grouping numeric data in a FAIR-compliant manner
- Visualizing key attributes

You can extend this notebook to deeper statistical analysis, machine learning, or integrate with other open datasets. For further work, check the metadata object for more fields, license, and documentation in the Croissant schema.